In [1]:
import pandas as pd
import numpy as np

PATH = "malnutrition_children_ethiopia.csv"
df = pd.read_csv(PATH)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nNutrition_Status distribution:")
print(df["Nutrition_Status"].value_counts(dropna=False))

df["target"] = (df["Nutrition_Status"] == "Malnourished").astype(int)

print("\nBinary target distribution (1=Malnourished):")
print(df["target"].value_counts())
print("\nPositive rate:", df["target"].mean())

na = df.isna().mean().sort_values(ascending=False)
print("\nMissing-value rate (top):")
print(na.head(10))

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumeric summary (selected):")
print(df[num_cols].describe().T[["min", "50%", "mean", "max"]])

if "ID" in df.columns:
    print("\nUnique IDs:", df["ID"].nunique(), "out of", len(df))


Shape: (4098, 16)

Columns: ['ID', 'Age (months)', 'Gender', 'Region', 'Mother_Education', 'Household_Wealth_Index', 'Height_cm', 'Weight_kg', 'Stunting', 'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB', 'Nutrition_Status']

Nutrition_Status distribution:
Nutrition_Status
Normal          2030
At_Risk         1238
Malnourished     830
Name: count, dtype: int64

Binary target distribution (1=Malnourished):
target
0    3268
1     830
Name: count, dtype: int64

Positive rate: 0.2025378233284529

Missing-value rate (top):
ID                        0.0
Age (months)              0.0
Gender                    0.0
Region                    0.0
Mother_Education          0.0
Household_Wealth_Index    0.0
Height_cm                 0.0
Weight_kg                 0.0
Stunting                  0.0
Underweight               0.0
dtype: float64

Numeric summary (selected):
               min     50%         mean     max
ID             1.0  2049.5  2049.500000  4098.0
Age (months)   0.0

In [2]:
from sklearn.model_selection import train_test_split
PATH = "malnutrition_children_ethiopia.csv"
df = pd.read_csv(PATH)

df["target"] = (df["Nutrition_Status"] == "Malnourished").astype(int)

df = df.drop(columns=["ID", "Nutrition_Status"])

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,      
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,      
    stratify=y_temp,
    random_state=42
)

print("Split sizes:")
print("Train:", X_train.shape, "Pos rate:", y_train.mean())
print("Val:  ", X_val.shape,   "Pos rate:", y_val.mean())
print("Test: ", X_test.shape,  "Pos rate:", y_test.mean())

def add_features(split: pd.DataFrame) -> pd.DataFrame:
    split = split.copy()
    split["BMI"] = split["Weight_kg"] / (split["Height_cm"] / 100.0) ** 2

    split["Weight_per_Month"] = split["Weight_kg"] / (split["Age (months)"] + 1)
    split["Height_per_Month"] = split["Height_cm"] / (split["Age (months)"] + 1)

    age_bins = [0, 6, 12, 24, 36, 48, 60]
    age_labels = ["0-6", "6-12", "12-24", "24-36", "36-48", "48-60"]
    split["Age_Group"] = pd.cut(
        split["Age (months)"],
        bins=age_bins,
        labels=age_labels,
        include_lowest=True
    )

    disease_cols = ["Anemia", "Malaria", "Diarrhea", "TB"]
    split["Disease_Count"] = split[disease_cols].sum(axis=1)
    split["Has_Multiple_Diseases"] = (split["Disease_Count"] >= 2).astype(int)

    nutrition_flags = ["Stunting", "Underweight", "Overweight"]
    split["Nutrition_Risk_Score"] = split[nutrition_flags].sum(axis=1)

    edu_map = {"None": 0, "Primary": 1, "Secondary": 2, "Higher": 3}
    wealth_map = {"Poorest": 0, "Poor": 1, "Middle": 2, "Rich": 3, "Richest": 4}

    split["Mother_Education_Ord"] = split["Mother_Education"].map(edu_map)
    split["Wealth_Ord"] = split["Household_Wealth_Index"].map(wealth_map)

    split = split.drop(columns=["Mother_Education", "Household_Wealth_Index"])

    return split


X_train_fe = add_features(X_train)
X_val_fe   = add_features(X_val)
X_test_fe  = add_features(X_test)

cat_cols = ["Gender", "Region", "Age_Group"]

X_train_fe = pd.get_dummies(X_train_fe, columns=cat_cols, drop_first=True)
X_val_fe   = pd.get_dummies(X_val_fe,   columns=cat_cols, drop_first=True)
X_test_fe  = pd.get_dummies(X_test_fe,  columns=cat_cols, drop_first=True)

X_train_fe, X_val_fe = X_train_fe.align(X_val_fe, join="left", axis=1, fill_value=0)
X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join="left", axis=1, fill_value=0)

print("\nAfter feature engineering:")
print("Train:", X_train_fe.shape, "Val:", X_val_fe.shape, "Test:", X_test_fe.shape)

print("NaN check (train/val/test):",
      X_train_fe.isna().sum().sum(),
      X_val_fe.isna().sum().sum(),
      X_test_fe.isna().sum().sum())

print("\nSample engineered columns:", [c for c in X_train_fe.columns if c in
      ["BMI", "Weight_per_Month", "Height_per_Month", "Disease_Count",
       "Has_Multiple_Diseases", "Nutrition_Risk_Score", "Mother_Education_Ord", "Wealth_Ord"]])


Split sizes:
Train: (2868, 14) Pos rate: 0.2025801952580195
Val:   (615, 14) Pos rate: 0.2032520325203252
Test:  (615, 14) Pos rate: 0.2016260162601626

After feature engineering:
Train: (2868, 28) Val: (615, 28) Test: (615, 28)
NaN check (train/val/test): 2649 568 545

Sample engineered columns: ['BMI', 'Weight_per_Month', 'Height_per_Month', 'Disease_Count', 'Has_Multiple_Diseases', 'Nutrition_Risk_Score', 'Mother_Education_Ord', 'Wealth_Ord']


In [3]:
from sklearn.impute import SimpleImputer

for col in ["Mother_Education_Ord", "Wealth_Ord"]:
    for split in [X_train_fe, X_val_fe, X_test_fe]:
        split[col] = split[col].fillna(0)  # lowest category = safest assumption


num_cols = X_train_fe.select_dtypes(include=["int64", "float64"]).columns

num_imputer = SimpleImputer(strategy="median")

X_train_fe[num_cols] = num_imputer.fit_transform(X_train_fe[num_cols])
X_val_fe[num_cols]   = num_imputer.transform(X_val_fe[num_cols])
X_test_fe[num_cols]  = num_imputer.transform(X_test_fe[num_cols])

print("NaNs after fix:",
      X_train_fe.isna().sum().sum(),
      X_val_fe.isna().sum().sum(),
      X_test_fe.isna().sum().sum())


NaNs after fix: 0 0 0


In [4]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report
)

In [5]:
def evaluate_model(name, model, X_tr, y_tr, X_val, y_val):
    model.fit(X_tr, y_tr)

    val_probs = model.predict_proba(X_val)[:, 1]

    roc = roc_auc_score(y_val, val_probs)
    pr  = average_precision_score(y_val, val_probs)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"ROC-AUC : {roc:.4f}")
    print(f"PR-AUC  : {pr:.4f}")

    return {
        "model": model,
        "roc_auc": roc,
        "pr_auc": pr,
        "val_probs": val_probs
    }

In [6]:
log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
    n_jobs=-1
)

logreg_results = evaluate_model(
    "Logistic Regression",
    log_reg,
    X_train_fe, y_train,
    X_val_fe, y_val
)


Logistic Regression
-------------------
ROC-AUC : 0.4897
PR-AUC  : 0.1899


In [7]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_results = evaluate_model(
    "Random Forest",
    rf,
    X_train_fe, y_train,
    X_val_fe, y_val
)


Random Forest
-------------
ROC-AUC : 0.5145
PR-AUC  : 0.2056


In [8]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[1, (len(y_train) - y_train.sum()) / y_train.sum()],
    verbose=0,
    random_seed=42
)

cat_results = evaluate_model(
    "CatBoost",
    cat,
    X_train_fe, y_train,
    X_val_fe, y_val
)


CatBoost
--------
ROC-AUC : 0.4864
PR-AUC  : 0.1940


In [9]:
from xgboost import XGBClassifier

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_results = evaluate_model(
    "XGBoost",
    xgb,
    X_train_fe, y_train,
    X_val_fe, y_val
)


XGBoost
-------
ROC-AUC : 0.5173
PR-AUC  : 0.2013


In [10]:
results_df = pd.DataFrame([
    {"Model": "Logistic Regression", "PR-AUC": logreg_results["pr_auc"]},
    {"Model": "Random Forest",       "PR-AUC": rf_results["pr_auc"]},
    {"Model": "CatBoost",            "PR-AUC": cat_results["pr_auc"]},
    {"Model": "XGBoost",             "PR-AUC": xgb_results["pr_auc"]},
]).sort_values("PR-AUC", ascending=False)

print(results_df)

                 Model    PR-AUC
1        Random Forest  0.205635
3              XGBoost  0.201277
2             CatBoost  0.193976
0  Logistic Regression  0.189897


In [11]:
from sklearn.metrics import accuracy_score

accuracy_results = []

logreg_val_preds = logreg_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val, logreg_val_preds)
})


rf_val_preds = rf_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val, rf_val_preds)
})


cat_val_preds = cat_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "CatBoost",
    "Accuracy": accuracy_score(y_val, cat_val_preds)
})


xgb_val_preds = xgb_results["model"].predict(X_val_fe)
accuracy_results.append({
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_val, xgb_val_preds)
})

accuracy_df = pd.DataFrame(accuracy_results).sort_values("Accuracy", ascending=False)
print(accuracy_df)

                 Model  Accuracy
3              XGBoost  0.700813
2             CatBoost  0.700813
1        Random Forest  0.699187
0  Logistic Regression  0.515447


In [12]:
rf_val_probs  = rf_results["model"].predict_proba(X_val_fe)[:, 1]
xgb_val_probs = xgb_results["model"].predict_proba(X_val_fe)[:, 1]
cat_val_probs = cat_results["model"].predict_proba(X_val_fe)[:, 1]

In [13]:
#baseline Ensemble
ensemble_val_probs = (
    rf_val_probs +
    xgb_val_probs +
    cat_val_probs
) / 3.0

In [14]:
#validation
from sklearn.metrics import roc_auc_score, average_precision_score

print("Ensemble ROC-AUC:", roc_auc_score(y_val, ensemble_val_probs))
print("Ensemble PR-AUC :", average_precision_score(y_val, ensemble_val_probs))

Ensemble ROC-AUC: 0.5040816326530612
Ensemble PR-AUC : 0.19711207784251278


In [15]:
#weighted soft voting
ensemble_val_probs_weighted = (
    0.4 * rf_val_probs +
    0.3 * xgb_val_probs +
    0.3 * cat_val_probs
)

In [16]:
print("Weighted Ensemble ROC-AUC:",
      roc_auc_score(y_val, ensemble_val_probs_weighted))
print("Weighted Ensemble PR-AUC :",
      average_precision_score(y_val, ensemble_val_probs_weighted))

Weighted Ensemble ROC-AUC: 0.5044081632653061
Weighted Ensemble PR-AUC : 0.19716280992286728


In [17]:
#Thereshold Optimization
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val, ensemble_val_probs_weighted
)

pr_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

candidate = pr_df[
    (pr_df["recall"] >= 0.70) &
    (pr_df["precision"] >= 0.21)
].head(1)

candidate

,threshold,precision,recall
73,0.210204,0.210332,0.912


In [18]:
#final eval
rf_test_probs  = rf_results["model"].predict_proba(X_test_fe)[:, 1]
xgb_test_probs = xgb_results["model"].predict_proba(X_test_fe)[:, 1]
cat_test_probs = cat_results["model"].predict_proba(X_test_fe)[:, 1]

ensemble_test_probs = (
    0.4 * rf_test_probs +
    0.3 * xgb_test_probs +
    0.3 * cat_test_probs
)

best_threshold = float(candidate["threshold"].values[0])

ensemble_test_preds = (ensemble_test_probs >= best_threshold).astype(int)


In [19]:
#metrics
from sklearn.metrics import classification_report, confusion_matrix

print("TEST ROC-AUC:",
      roc_auc_score(y_test, ensemble_test_probs))
print("TEST PR-AUC :",
      average_precision_score(y_test, ensemble_test_probs))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ensemble_test_preds))

print("\nClassification Report:")
print(classification_report(y_test, ensemble_test_preds, digits=3))

TEST ROC-AUC: 0.5049109782537284
TEST PR-AUC : 0.20515414120213876

Confusion Matrix:
[[ 80 411]
 [ 14 110]]

Classification Report:
              precision    recall  f1-score   support

           0      0.851     0.163     0.274       491
           1      0.211     0.887     0.341       124

    accuracy                          0.309       615
   macro avg      0.531     0.525     0.307       615
weighted avg      0.722     0.309     0.287       615



In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Validation predictions
ensemble_val_preds = (ensemble_val_probs_weighted >= best_threshold).astype(int)

val_metrics = {
    "Accuracy": accuracy_score(y_val, ensemble_val_preds),
    "Precision": precision_score(y_val, ensemble_val_preds),
    "Recall": recall_score(y_val, ensemble_val_preds),
    "F1-score": f1_score(y_val, ensemble_val_preds)
}

print("Validation Metrics (Ensemble):")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}")


Validation Metrics (Ensemble):
Accuracy: 0.2862
Precision: 0.2103
Recall: 0.9120
F1-score: 0.3418


In [21]:
# Test predictions
ensemble_test_preds = (ensemble_test_probs >= best_threshold).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, ensemble_test_preds),
    "Precision": precision_score(y_test, ensemble_test_preds),
    "Recall": recall_score(y_test, ensemble_test_preds),
    "F1-score": f1_score(y_test, ensemble_test_preds)
}

print("\nTest Metrics (Ensemble):")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")


Test Metrics (Ensemble):
Accuracy: 0.3089
Precision: 0.2111
Recall: 0.8871
F1-score: 0.3411


In [22]:
def get_metrics(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds),
        "Recall": recall_score(y_true, preds),
        "F1-score": f1_score(y_true, preds)
    }

comparison = pd.DataFrame.from_dict({
    "Random Forest": get_metrics(y_val, rf_val_probs, best_threshold),
    "XGBoost":       get_metrics(y_val, xgb_val_probs, best_threshold),
    "CatBoost":      get_metrics(y_val, cat_val_probs, best_threshold),
    "Ensemble":      get_metrics(y_val, ensemble_val_probs_weighted, best_threshold)
}, orient="index")

print(comparison)

               Accuracy  Precision  Recall  F1-score
Random Forest  0.203252   0.203252   1.000  0.337838
XGBoost        0.497561   0.230994   0.632  0.338330
CatBoost       0.437398   0.202156   0.600  0.302419
Ensemble       0.286179   0.210332   0.912  0.341829


In [23]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.num_imputer = SimpleImputer(strategy="median")
        self.final_columns_ = None

        self.edu_map = {"None": 0, "Primary": 1, "Secondary": 2, "Higher": 3}
        self.wealth_map = {"Poorest": 0, "Poor": 1, "Middle": 2, "Rich": 3, "Richest": 4}

        self.cat_cols = ["Gender", "Region", "Age_Group"]
        self.disease_cols = ["Anemia", "Malaria", "Diarrhea", "TB"]
        self.nutrition_flags = ["Stunting", "Underweight", "Overweight"]

    def _feature_engineering(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        # BMI + safe ratios
        X["BMI"] = X["Weight_kg"] / (X["Height_cm"] / 100.0) ** 2
        X["Weight_per_Month"] = X["Weight_kg"] / (X["Age (months)"] + 1)
        X["Height_per_Month"] = X["Height_cm"] / (X["Age (months)"] + 1)

        # Age bins
        age_bins = [0, 6, 12, 24, 36, 48, 60]
        age_labels = ["0-6", "6-12", "12-24", "24-36", "36-48", "48-60"]
        X["Age_Group"] = pd.cut(
            X["Age (months)"], bins=age_bins, labels=age_labels, include_lowest=True
        ).astype("object").fillna("Unknown")

        # Disease + nutrition risk scores
        X["Disease_Count"] = X[self.disease_cols].sum(axis=1)
        X["Has_Multiple_Diseases"] = (X["Disease_Count"] >= 2).astype(int)
        X["Nutrition_Risk_Score"] = X[self.nutrition_flags].sum(axis=1)

        # Ordinal encoding with safe fallback
        X["Mother_Education_Ord"] = X["Mother_Education"].map(self.edu_map).fillna(0)
        X["Wealth_Ord"] = X["Household_Wealth_Index"].map(self.wealth_map).fillna(0)

        # Drop original ordinal columns
        X = X.drop(columns=["Mother_Education", "Household_Wealth_Index"])

        return X

    def fit(self, X: pd.DataFrame, y=None):
        X = self._feature_engineering(X)

        # One-hot nominal categoricals
        X = pd.get_dummies(X, columns=self.cat_cols, drop_first=True)

        # Freeze final schema
        self.final_columns_ = X.columns.tolist()

        # 🔒 Freeze numeric columns EXACTLY as seen during fit
        self.num_cols_ = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

        # Fit imputer ONLY on frozen numeric columns
        self.num_imputer.fit(X[self.num_cols_])

        return self
        
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = self._feature_engineering(X)

        # One-hot encode
        X = pd.get_dummies(X, columns=self.cat_cols, drop_first=True)

        # 🔑 ALIGN FIRST
        for col in self.final_columns_:
            if col not in X.columns:
                X[col] = 0

        X = X[self.final_columns_]

        # 🔒 Use EXACT numeric columns seen during fit
        X[self.num_cols_] = self.num_imputer.transform(X[self.num_cols_])

        return X




In [24]:
# Build preprocessor from RAW train split (not X_train_fe)
pre = Preprocessor()
pre.fit(X_train)

X_train_pp = pre.transform(X_train)
X_val_pp   = pre.transform(X_val)
X_test_pp  = pre.transform(X_test)

print("Shapes:", X_train_pp.shape, X_val_pp.shape, X_test_pp.shape)
print("NaNs:", X_train_pp.isna().sum().sum(), X_val_pp.isna().sum().sum(), X_test_pp.isna().sum().sum())
print("First 10 columns:", X_train_pp.columns[:10].tolist())

Shapes: (2868, 28) (615, 28) (615, 28)
NaNs: 0 0 0
First 10 columns: ['Age (months)', 'Height_cm', 'Weight_kg', 'Stunting', 'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB']


In [25]:
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Class imbalance ratio
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

# -----------------
# Random Forest
# -----------------
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_pp, y_train)

# -----------------
# XGBoost
# -----------------
xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_pp, y_train)

# -----------------
# CatBoost
# -----------------
cat = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[1, pos_weight],
    random_seed=42,
    verbose=0
)
cat.fit(X_train_pp, y_train)

In [26]:
from sklearn.metrics import average_precision_score

rf_val_probs  = rf.predict_proba(X_val_pp)[:, 1]
xgb_val_probs = xgb.predict_proba(X_val_pp)[:, 1]
cat_val_probs = cat.predict_proba(X_val_pp)[:, 1]

print("Validation PR-AUCs:")
print("Random Forest:", average_precision_score(y_val, rf_val_probs))
print("XGBoost:      ", average_precision_score(y_val, xgb_val_probs))
print("CatBoost:     ", average_precision_score(y_val, cat_val_probs))

Validation PR-AUCs:
Random Forest: 0.20498380571987004
XGBoost:       0.20445748817812748
CatBoost:      0.2007511027075875


In [27]:
import numpy as np

class SoftVotingEnsemble:
    def __init__(self, rf, xgb, cat, weights=(0.4, 0.3, 0.3), threshold=0.5):
        self.rf = rf
        self.xgb = xgb
        self.cat = cat
        self.weights = np.array(weights, dtype=float)
        self.threshold = float(threshold)

    def predict_proba(self, X):
        rf_p  = self.rf.predict_proba(X)[:, 1]
        xgb_p = self.xgb.predict_proba(X)[:, 1]
        cat_p = self.cat.predict_proba(X)[:, 1]

        w = self.weights / self.weights.sum()
        p = w[0]*rf_p + w[1]*xgb_p + w[2]*cat_p

        return np.vstack([1 - p, p]).T

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold).astype(int)

In [28]:
ensemble = SoftVotingEnsemble(
    rf=rf,
    xgb=xgb,
    cat=cat,
    weights=(0.4, 0.3, 0.3),
    threshold=0.5  # temporary
)

In [29]:
from sklearn.metrics import average_precision_score

ensemble_val_probs = ensemble.predict_proba(X_val_pp)[:, 1]

print("Ensemble Validation PR-AUC:",
      average_precision_score(y_val, ensemble_val_probs))

Ensemble Validation PR-AUC: 0.20231407937066392


In [30]:
from sklearn.metrics import precision_recall_curve
import pandas as pd

precision, recall, thresholds = precision_recall_curve(
    y_val, ensemble_val_probs
)

pr_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

# Choose threshold prioritizing recall (screening use-case)
candidates = pr_df[
    (pr_df["recall"] >= 0.85) & (pr_df["precision"] >= 0.20)
]

candidates.head()

,threshold,precision,recall
0,0.101642,0.203252,1.0
1,0.103167,0.203583,1.0
2,0.105068,0.203915,1.0
3,0.127286,0.204248,1.0
4,0.127576,0.204583,1.0


In [31]:
best_threshold = float(candidates.iloc[0]["threshold"])
print("Chosen threshold:", best_threshold)

ensemble.threshold = best_threshold

Chosen threshold: 0.10164208190281886


In [32]:
print("Ensemble threshold:", ensemble.threshold)
print("Ensemble weights:", ensemble.weights)

Ensemble threshold: 0.10164208190281886
Ensemble weights: [0.4 0.3 0.3]


In [33]:
import joblib
import json

# 1️⃣ Save preprocessor
joblib.dump(pre, "preprocessor.pkl")

# 2️⃣ Save ensemble
joblib.dump(ensemble, "ensemble.pkl")

# 3️⃣ Save metadata (explicit & auditable)
metadata = {
    "model_type": "SoftVotingEnsemble",
    "base_models": ["RandomForest", "XGBoost", "CatBoost"],
    "weights": ensemble.weights.tolist(),
    "threshold": ensemble.threshold,
    "objective": "High-recall malnutrition screening",
    "target_definition": "1 = Malnourished, 0 = Normal/At_Risk",
    "features_count": len(pre.final_columns_),
}

with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Artifacts saved:")
print("- preprocessor.pkl")
print("- ensemble.pkl")
print("- metadata.json")

Artifacts saved:
- preprocessor.pkl
- ensemble.pkl
- metadata.json


In [34]:
# Reload artifacts
pre_loaded = joblib.load("preprocessor.pkl")
ensemble_loaded = joblib.load("ensemble.pkl")

# Transform raw test data again
X_test_pp_loaded = pre_loaded.transform(X_test)

# Predict using loaded ensemble
loaded_probs = ensemble_loaded.predict_proba(X_test_pp_loaded)[:, 1]
loaded_preds = ensemble_loaded.predict(X_test_pp_loaded)

# Compare with original ensemble
original_probs = ensemble.predict_proba(X_test_pp)[:, 1]
original_preds = ensemble.predict(X_test_pp)

print("Probability difference (max):",
      abs(original_probs - loaded_probs).max())

print("Prediction consistency:",
      (original_preds == loaded_preds).all())

Probability difference (max): 2.220446049250313e-16
Prediction consistency: True


In [35]:
import pandas as pd

WHO_XLSX_PATH = "WHO_Recommendations_binary.xlsx"

who_df = pd.read_excel(WHO_XLSX_PATH)

print("WHO sheet shape:", who_df.shape)
print("Columns:", who_df.columns.tolist())
display(who_df.head(10))


WHO sheet shape: (10, 4)
Columns: ['Feature', 'Malnourished Child – WHO-Informed Recommendation', 'Well-Nourished Child – WHO-Informed Recommendation', 'WHO Reference']


,Feature,Malnourished Child – WHO-Informed Recommendation,Well-Nourished Child – WHO-Informed Recommendation,WHO Reference
0,Low weight (Wasting),Assess severity using WHO Child Growth Standar...,Continue age-appropriate balanced diet; routin...,WHO SAM Guidelines (2013)
1,Low height (Stunting),Provide long-term nutrition support; improve d...,Maintain adequate nutrition and health practic...,WHO Essential Nutrition Actions(2013); WHO Mal...
2,Underweight,"Identify underlying causes (poor intake, infec...",Maintain balanced diet and preventive healthca...,WHO Management of Moderate Acute Malnutrition;...
3,Stunting (Chronic undernutrition),"Promote optimal IYCF, maternal nutrition, dise...","Sustain appropriate feeding practices, materna...",WHO & UNICEF Global Strategy for IYCF
4,Tuberculosis (TB),Diagnose and start prompt anti-TB treatment ac...,Standard pediatric TB management per WHO guide...,WHO Consolidated Guidelines on TB: Module 5 — ...
5,Anaemia,Provide iron supplementation per age group; di...,Preventive nutrition counselling; periodic hae...,WHO Daily Iron Supplementation Guidelines; WHO...
6,Diarrhea,Manage with ORS and zinc supplementation; cont...,"Promote hygiene, safe water, and sanitation; c...",WHO & UNICEF Diarrhoea Management Guidelines
7,Malaria,Prompt diagnosis and treatment according to ma...,"Malaria prevention (bed nets, vector control);...",WHO Malaria Treatment Guidelines
8,Low household wealth,"Prioritize social protection, food assistance ...",Continue preventive nutrition services and gro...,WHO Malnutrition Fact Sheet; WHO Essential Nut...
9,Low maternal education,"Intensive caregiver education on feeding, hygi...",Ongoing health education reinforcement; empowe...,WHO Essential Nutrition Actions(2013)


In [36]:
def normalize_who_sheet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalize column names
    df.columns = [c.strip() for c in df.columns]

    # Normalize Feature field
    df["Feature"] = df["Feature"].astype(str).str.strip()

    # Rename recommendation columns to stable internal keys
    df = df.rename(columns={
        "Malnourished Child – WHO-Informed Recommendation": "rec_malnourished",
        "Well-Nourished Child – WHO-Informed Recommendation": "rec_well",
        "WHO Reference": "who_reference"
    })

    return df

who_df = normalize_who_sheet(who_df)

In [37]:
import numpy as np

def compute_clinical_features(payload: dict) -> dict:
    """
    payload: raw input from UI (same 14 fields)
    returns: engineered clinical fields needed for rules
    """
    age = float(payload["Age (months)"])
    height_cm = float(payload["Height_cm"])
    weight_kg = float(payload["Weight_kg"])

    # Safe BMI
    height_m = height_cm / 100.0
    bmi = weight_kg / (height_m ** 2) if height_m > 0 else np.nan

    disease_count = int(payload["Anemia"]) + int(payload["Malaria"]) + int(payload["Diarrhea"]) + int(payload["TB"])
    has_multiple_diseases = int(disease_count >= 2)

    nutrition_risk_score = int(payload["Stunting"]) + int(payload["Underweight"]) + int(payload["Overweight"])

    return {
        "BMI": bmi,
        "Disease_Count": disease_count,
        "Has_Multiple_Diseases": has_multiple_diseases,
        "Nutrition_Risk_Score": nutrition_risk_score
    }

In [38]:
CLINICAL_THRESHOLDS = {
    "BMI_WASTING_CUTOFF": 14.0
}

def get_trigger_map(thresholds=CLINICAL_THRESHOLDS):
    return {
        "Low weight (Wasting)": lambda p, c: c["BMI"] < thresholds["BMI_WASTING_CUTOFF"],
        "Low height (Stunting)": lambda p, c: int(p["Stunting"]) == 1,
        "Underweight": lambda p, c: int(p["Underweight"]) == 1,
        "Overweight": lambda p, c: int(p["Overweight"]) == 1,
        "Anemia": lambda p, c: int(p["Anemia"]) == 1,
        "Malaria": lambda p, c: int(p["Malaria"]) == 1,
        "Diarrhea": lambda p, c: int(p["Diarrhea"]) == 1,
        "Tuberculosis (TB)": lambda p, c: int(p["TB"]) == 1,
        "Multiple illnesses": lambda p, c: c["Disease_Count"] >= 2,
        "General malnutrition risk": lambda p, c: True  # always include base guidance
    }

In [39]:
def generate_recommendations(payload: dict, risk_label: str, who_sheet: pd.DataFrame) -> list[dict]:
    """
    risk_label: "HIGH" or "LOW"
    Returns: list of {feature, recommendation, who_reference}
    """
    clinical = compute_clinical_features(payload)
    trigger_map = get_trigger_map()

    # Choose column based on risk label
    rec_col = "rec_malnourished" if risk_label == "HIGH" else "rec_well"

    results = []
    for _, row in who_sheet.iterrows():
        feature_name = row["Feature"]

        trigger_fn = trigger_map.get(feature_name)
        if trigger_fn is None:
            # If sheet includes a feature we didn't map yet, skip safely (no crash)
            continue

        if trigger_fn(payload, clinical):
            rec_text = str(row.get(rec_col, "")).strip()
            who_ref = str(row.get("who_reference", "")).strip()

            if rec_text:  # include only non-empty recommendations
                results.append({
                    "feature": feature_name,
                    "recommendation": rec_text,
                    "who_reference": who_ref
                })

    # Deduplicate recommendations (by feature + text)
    dedup = {}
    for r in results:
        key = (r["feature"], r["recommendation"])
        dedup[key] = r

    return list(dedup.values())


In [40]:
sample_payload = X_test.iloc[0].to_dict()

X_sample_pp = pre.transform(pd.DataFrame([sample_payload]))
prob = float(ensemble.predict_proba(X_sample_pp)[0, 1])
risk_label = "HIGH" if prob >= ensemble.threshold else "LOW"

recs = generate_recommendations(sample_payload, risk_label, who_df)

print("Risk:", risk_label, "Prob:", prob)
print("Recommendations count:", len(recs))
for r in recs:
    print("-", r["feature"], "=>", r["recommendation"])



Risk: HIGH Prob: 0.3228759526034263
Recommendations count: 2
- Diarrhea => Manage with ORS and zinc supplementation; continue feeding during illness; monitor weight to prevent acute malnutrition
- Malaria => Prompt diagnosis and treatment according to malaria guidelines; integrate nutrition rehabilitation during recovery


In [41]:
REQUIRED_FIELDS = [
    "Age (months)", "Gender",
    "Mother_Education", "Household_Wealth_Index",
    "Height_cm", "Weight_kg",
    "Stunting", "Underweight", "Overweight",
    "Anemia", "Malaria", "Diarrhea", "TB"
]

In [42]:
import random

ETHIOPIA_REGIONS = [
    "Addis Ababa","Afar","Amhara","Benishangul-Gumuz","Dire Dawa","Gambela",
    "Harari","Oromia","Sidama","Somali","South West Ethiopia","Southern Nations","Tigray"
]

def inject_region(payload: dict) -> dict:
    payload = payload.copy()
    payload["Region"] = random.choice(ETHIOPIA_REGIONS)
    return payload

In [43]:
def build_top_factors(payload: dict, clinical: dict, top_k: int = 5) -> list[dict]:
    factors = []

    # BMI-based factor
    if not np.isnan(clinical["BMI"]):
        if clinical["BMI"] < 14.0:
            factors.append({"feature": "BMI", "value": round(clinical["BMI"], 2), "direction": "increases_risk"})

    # Nutrition flags
    for f in ["Stunting", "Underweight", "Overweight"]:
        if int(payload.get(f, 0)) == 1:
            factors.append({"feature": f, "value": 1, "direction": "increases_risk"})

    # Diseases
    for d in ["Anemia", "Malaria", "Diarrhea", "TB"]:
        if int(payload.get(d, 0)) == 1:
            factors.append({"feature": d, "value": 1, "direction": "increases_risk"})

    # Aggregate burden
    factors.append({"feature": "Disease_Count", "value": int(clinical["Disease_Count"]), "direction": "increases_risk" if clinical["Disease_Count"] >= 2 else "neutral"})
    factors.append({"feature": "Nutrition_Risk_Score", "value": int(clinical["Nutrition_Risk_Score"]), "direction": "increases_risk" if clinical["Nutrition_Risk_Score"] >= 1 else "neutral"})

    # Rank factors: prioritize active issues
    def score(item):
        if item["direction"] == "increases_risk":
            return 2
        if item["direction"] == "neutral":
            return 1
        return 0

    factors = sorted(factors, key=score, reverse=True)
    return factors[:top_k]

In [44]:
import pandas as pd
import numpy as np

def validate_payload(payload: dict):
    missing = [k for k in REQUIRED_FIELDS if k not in payload]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")

def predict_and_recommend(payload: dict, pre, ensemble, who_df) -> dict:
    # 1) Validate
    validate_payload(payload)

    # 2) Inject Region (backend-only)
    payload_model = inject_region(payload)

    # 3) ML inference
    X_raw = pd.DataFrame([payload_model])
    X_pp = pre.transform(X_raw)

    risk_probability = float(ensemble.predict_proba(X_pp)[0, 1])
    risk_label = "HIGH" if risk_probability >= ensemble.threshold else "LOW"

    # 4) Decision-support triggers
    clinical = compute_clinical_features(payload_model)

    # 5) Recommendations from spreadsheet
    recommendations = generate_recommendations(payload_model, risk_label, who_df)

    # 6) Explanations (XAI Hook v1)
    top_factors = build_top_factors(payload_model, clinical, top_k=5)

    # 7) Unified response contract
    return {
        "risk_probability": risk_probability,
        "risk_label": risk_label,
        "threshold": float(ensemble.threshold),

        # Explanations (hook v1 - clinical factors)
        "explanations": {
            "type": "clinical_factors_v1",   # later: shap_v1
            "top_factors": top_factors
        },

        # Recommendations
        "recommendations": recommendations,

        # Internal fields (for logs / debugging)
        "debug": {
            "region_used_for_model": payload_model["Region"]
        }
    }

In [45]:
sample_payload = X_test.iloc[0].to_dict()  # raw without Region (OK)

response = predict_and_recommend(sample_payload, pre, ensemble, who_df)

print("RISK:", response["risk_label"], "PROB:", response["risk_probability"])
print("TOP FACTORS:", response["explanations"]["top_factors"])
print("RECS:", len(response["recommendations"]))
for r in response["recommendations"]:
    print("-", r["feature"], "=>", r["recommendation"])
print("DEBUG region:", response["debug"]["region_used_for_model"])

RISK: HIGH PROB: 0.32287595260342616
TOP FACTORS: [{'feature': 'Overweight', 'value': 1, 'direction': 'increases_risk'}, {'feature': 'Anemia', 'value': 1, 'direction': 'increases_risk'}, {'feature': 'Malaria', 'value': 1, 'direction': 'increases_risk'}, {'feature': 'Diarrhea', 'value': 1, 'direction': 'increases_risk'}, {'feature': 'Disease_Count', 'value': 3, 'direction': 'increases_risk'}]
RECS: 2
- Diarrhea => Manage with ORS and zinc supplementation; continue feeding during illness; monitor weight to prevent acute malnutrition
- Malaria => Prompt diagnosis and treatment according to malaria guidelines; integrate nutrition rehabilitation during recovery
DEBUG region: Southern Nations


In [46]:
# If SHAP is not installed in your venv, uncomment:
# !pip install shap

import shap
import numpy as np
import pandas as pd


In [47]:
# Background sample for SHAP
BACKGROUND_SIZE = 200
background = X_train_pp.sample(n=min(BACKGROUND_SIZE, len(X_train_pp)), random_state=42)

In [48]:
# Make SHAP-safe background (ALL numeric, float)
background_shap = background.astype(float)


In [49]:
# --- Random Forest explainer ---
rf_explainer = shap.TreeExplainer(rf, data=background_shap)

# --- XGBoost explainer ---
try:
    xgb_explainer = shap.TreeExplainer(
        xgb,
        data=background_shap,
        model_output="probability"
    )
except TypeError:
    xgb_explainer = shap.TreeExplainer(xgb, data=background_shap)

# --- CatBoost explainer ---
try:
    cat_explainer = shap.TreeExplainer(cat, data=background_shap)
    CAT_SHAP_MODE = "shap"
except Exception as e:
    print("CatBoost SHAP TreeExplainer failed, using native CatBoost SHAP")
    CAT_SHAP_MODE = "native"
    cat_explainer = None


In [50]:
def shap_values_one(explainer, X_one: pd.DataFrame):
    """
    Returns SHAP values for POSITIVE CLASS (class 1) as 1D array.
    Handles sklearn RF binary-class doubling issue.
    """
    sv = explainer.shap_values(X_one)

    if isinstance(sv, list):
        # Typical [class0, class1]
        sv = sv[1]
    else:
        sv = np.array(sv)

    sv = sv.reshape(-1)

    n_features = X_one.shape[1]

    # 🔑 FIX: RF may return 2*n_features
    if sv.shape[0] == 2 * n_features:
        sv = sv[-n_features:]

    return sv



In [51]:
def catboost_native_shap(cat_model, X_one: pd.DataFrame) -> np.ndarray:
    """
    CatBoost native SHAP returns array with extra last column = expected value.
    We return feature SHAP values only.
    """
    # CatBoost returns (n_samples, n_features + 1)
    sv = cat_model.get_feature_importance(
        data=X_one,
        type="ShapValues"
    )
    sv = np.array(sv)
    return sv[0, :-1]  # drop expected value column


In [52]:
ENSEMBLE_WEIGHTS = np.array([0.4, 0.3, 0.3], dtype=float)  # RF, XGB, CAT

def ensemble_shap_explain(payload: dict, pre, rf, xgb, cat, top_k=5):
    """
    Returns top SHAP factors for one input payload using ensemble-weighted SHAP aggregation.
    """
    # Inject Region exactly like production inference (important for consistency)
    payload_model = inject_region(payload)

    # Preprocess to model feature space
    X_one = pre.transform(pd.DataFrame([payload_model]))

    # SHAP per model
    rf_sv = shap_values_one(rf_explainer, X_one)
    xgb_sv = shap_values_one(xgb_explainer, X_one)

    if CAT_SHAP_MODE == "shap":
        cat_sv = shap_values_one(cat_explainer, X_one)
    else:
        cat_sv = catboost_native_shap(cat, X_one)

    # Weighted aggregation (same weights as ensemble)
    w = ENSEMBLE_WEIGHTS / ENSEMBLE_WEIGHTS.sum()
    ens_sv = w[0]*rf_sv + w[1]*xgb_sv + w[2]*cat_sv

    # Build ranked top factors by absolute impact
    feature_names = X_one.columns.tolist()
    order = np.argsort(np.abs(ens_sv))[::-1][:top_k]

    top_factors = []
    for idx in order:
        name = feature_names[idx]
        val = float(ens_sv[idx])
        direction = "increases_risk" if val > 0 else "decreases_risk"
        top_factors.append({
            "feature": name,
            "shap_value": round(val, 6),
            "direction": direction
        })

    return top_factors


In [53]:
def predict_and_recommend_shap(payload: dict, pre, ensemble, who_df) -> dict:
    validate_payload(payload)

    # Region injected consistently
    payload_model = inject_region(payload)

    # ML inference
    X_raw = pd.DataFrame([payload_model])
    X_pp = pre.transform(X_raw)

    risk_probability = float(ensemble.predict_proba(X_pp)[0, 1])
    risk_label = "HIGH" if risk_probability >= ensemble.threshold else "LOW"

    # Recommendations
    recommendations = generate_recommendations(payload_model, risk_label, who_df)

    # SHAP explanations (true model-based XAI)
    top_factors = ensemble_shap_explain(payload, pre, rf, xgb, cat, top_k=5)

    return {
        "risk_probability": risk_probability,
        "risk_label": risk_label,
        "threshold": float(ensemble.threshold),
        "explanations": {
            "type": "shap_v1",
            "top_factors": top_factors
        },
        "recommendations": recommendations,
        "debug": {
            "region_used_for_model": payload_model["Region"]
        }
    }


In [54]:
sample_payload = X_test.iloc[0].to_dict()

response = predict_and_recommend_shap(sample_payload, pre, ensemble, who_df)

print("RISK:", response["risk_label"], "PROB:", response["risk_probability"])
print("EXPLANATIONS TYPE:", response["explanations"]["type"])
print("TOP SHAP FACTORS:")
for f in response["explanations"]["top_factors"]:
    print("-", f["feature"], f["shap_value"], f["direction"])

print("RECS:", len(response["recommendations"]))
for r in response["recommendations"]:
    print("-", r["feature"], "=>", r["recommendation"])


RISK: HIGH PROB: 0.3228759526034262
EXPLANATIONS TYPE: shap_v1
TOP SHAP FACTORS:
- Weight_kg -0.158228 decreases_risk
- Age (months) -0.113441 decreases_risk
- BMI 0.106622 increases_risk
- Height_per_Month 0.077671 increases_risk
- Wealth_Ord -0.055789 decreases_risk
RECS: 2
- Diarrhea => Manage with ORS and zinc supplementation; continue feeding during illness; monitor weight to prevent acute malnutrition
- Malaria => Prompt diagnosis and treatment according to malaria guidelines; integrate nutrition rehabilitation during recovery


In [56]:
from ml_core.preprocessing import Preprocessor
import joblib

pre = Preprocessor()
pre.fit(X_train)

joblib.dump(pre, "preprocessor.pkl")
joblib.dump(ensemble, "ensemble.pkl")

['ensemble.pkl']

In [57]:
import sklearn
print(sklearn.__version__)


1.7.2


In [58]:
from ml_core.ensemble import SoftVotingEnsemble
import joblib

ensemble = SoftVotingEnsemble(rf, xgb, cat, threshold=best_threshold)

joblib.dump(ensemble, "ensemble.pkl")

['ensemble.pkl']

In [59]:
import xgboost, catboost
print(xgboost.__version__)
print(catboost.__version__)


3.1.2
1.2.8


In [60]:
# Use raw training data BEFORE preprocessing
background = X_train.sample(n=100, random_state=42)

background.to_csv("background.csv", index=False)